# DA6401 Report - Section 2.9 Weight Initialization & Symmetry

Controlled comparison:
- `zeros` initialization
- `xavier` initialization

Track gradients of 5 neurons from the same hidden layer over first 50 training iterations.


In [ ]:
from __future__ import annotations

import json
from argparse import Namespace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ann.neural_network import NeuralNetwork
from utils.data_loader import load_dataset

print('Project root:', ROOT)


In [ ]:
RUN_EXPERIMENT = False  # Set True to run zeros/xavier experiments and log summary run.

PROJECT = 'da6401_assignment_1'
ENTITY = 'da25s007-'
SEED = 42
DATASET = 'mnist'
EPOCHS = 10
BATCH_SIZE = 64
TRACK_STEPS = 50
TRACK_NEURONS = 5
OUT_DIR = ROOT / 'src' / 'tmp'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def make_args(weight_init: str) -> Namespace:
    return Namespace(
        dataset=DATASET,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        loss='cross_entropy',
        optimizer='sgd',
        learning_rate=0.01,
        weight_decay=0.0,
        num_layers=2,
        hidden_size=[64, 64],
        activation='sigmoid',
        weight_init=weight_init,
        seed=SEED,
    )


def run_experiment(weight_init: str, data: dict):
    args = make_args(weight_init)
    model = NeuralNetwork(args)

    run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=f'report_2_9_{weight_init}',
        tags=['report', '2.9', 'weight_init', weight_init, 'symmetry'],
        config={
            'dataset': DATASET,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'loss': 'cross_entropy',
            'optimizer': 'sgd',
            'learning_rate': 0.01,
            'weight_init': weight_init,
            'activation': 'sigmoid',
            'num_layers': 2,
            'hidden_size': [64, 64],
            'track_steps': TRACK_STEPS,
            'track_neurons': TRACK_NEURONS,
            'tracked_layer': 0,
        },
    )

    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
    X_test, y_test = data['X_test'], data['y_test']
    y_train_onehot = model._to_onehot(y_train, model.output_dim)

    n_samples = X_train.shape[0]
    step_records = []
    epoch_records = []
    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        indices = model.rng.permutation(n_samples)
        batch_losses = []

        for start in range(0, n_samples, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n_samples)
            batch_idx = indices[start:end]

            X_batch = X_train[batch_idx]
            y_batch = y_train_onehot[batch_idx]

            logits = model.forward(X_batch)
            model.backward(y_batch, logits)

            grad_vec = model.layers[0].grad_b.reshape(-1)
            tracked = grad_vec[:TRACK_NEURONS].astype(float)
            grad_std = float(np.std(tracked))

            if global_step < TRACK_STEPS:
                rec = {
                    'step': global_step,
                    'init': weight_init,
                    'train/loss_step': float(model.last_loss),
                    'analysis/grad_std_5_neurons': grad_std,
                }
                for i in range(TRACK_NEURONS):
                    rec[f'analysis/grad_b_l0_n{i}'] = float(tracked[i])
                step_records.append(rec)

                run.log(
                    {k: v for k, v in rec.items() if k not in {'step', 'init'}},
                    step=global_step,
                )

            model.update_weights()
            if model.last_loss is not None:
                batch_losses.append(float(model.last_loss))

            global_step += 1

        val_metrics = model.evaluate(X_val, y_val, batch_size=1024)
        test_metrics = model.evaluate(X_test, y_test, batch_size=1024)

        row = {
            'epoch': epoch,
            'train_loss_step_mean': float(np.mean(batch_losses)) if batch_losses else float('nan'),
            'val_accuracy': float(val_metrics['accuracy']),
            'val_f1': float(val_metrics['f1']),
            'test_accuracy': float(test_metrics['accuracy']),
            'test_f1': float(test_metrics['f1']),
        }
        epoch_records.append(row)

        run.log(
            {
                'epoch': epoch,
                'train/loss': row['train_loss_step_mean'],
                'val/accuracy': row['val_accuracy'],
                'val/f1': row['val_f1'],
                'test/accuracy': row['test_accuracy'],
                'test/f1': row['test_f1'],
            },
            step=global_step,
        )

    final_val = model.evaluate(X_val, y_val, batch_size=1024)
    final_test = model.evaluate(X_test, y_test, batch_size=1024)
    run.summary['best/val_f1'] = float(final_val['f1'])
    run.summary['best/test_f1'] = float(final_test['f1'])
    run.summary['analysis/grad_std_mean_first50'] = float(
        np.mean([r['analysis/grad_std_5_neurons'] for r in step_records])
    )
    run.finish()

    return {
        'run_id': run.id,
        'run_name': run.name,
        'weight_init': weight_init,
        'step_records': step_records,
        'epoch_records': epoch_records,
        'final_val': {k: float(v) for k, v in final_val.items() if k != 'logits'},
        'final_test': {k: float(v) for k, v in final_test.items() if k != 'logits'},
    }


def make_plots(results: dict):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
    for ax, init_name in zip(axes, ['zeros', 'xavier']):
        rows = sorted(results[init_name]['step_records'], key=lambda x: x['step'])
        steps = np.array([r['step'] for r in rows], dtype=int)
        for i in range(TRACK_NEURONS):
            vals = np.array([r[f'analysis/grad_b_l0_n{i}'] for r in rows], dtype=float)
            ax.plot(steps, vals, label=f'Neuron {i}')
        ax.set_title(f'{init_name.title()} Init')
        ax.set_xlabel('Training iteration (step)')
        ax.set_ylabel('Gradient (first hidden-layer bias)')
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', fontsize=8)

    fig.suptitle('2.9: Gradient traces of 5 neurons over first 50 steps', fontsize=13)
    fig.tight_layout()
    grad_plot_path = OUT_DIR / 'report_2_9_gradient_traces.png'
    fig.savefig(grad_plot_path, dpi=160, bbox_inches='tight')
    plt.close(fig)

    fig2, ax2 = plt.subplots(figsize=(8, 5))
    for init_name in ['zeros', 'xavier']:
        rows = results[init_name]['epoch_records']
        ax2.plot([r['epoch'] for r in rows], [r['val_accuracy'] for r in rows], marker='o', label=init_name)
    ax2.set_title('2.9: Validation accuracy (zeros vs xavier)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation accuracy')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    val_plot_path = OUT_DIR / 'report_2_9_val_accuracy.png'
    fig2.savefig(val_plot_path, dpi=160, bbox_inches='tight')
    plt.close(fig2)

    return grad_plot_path, val_plot_path


def log_summary(results: dict, grad_plot_path: Path, val_plot_path: Path):
    run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name='report_2_9_summary',
        tags=['report', '2.9', 'weight_init', 'symmetry', 'summary'],
    )

    table = wandb.Table(columns=['init', 'step', 'n0', 'n1', 'n2', 'n3', 'n4', 'grad_std', 'train_loss_step'])
    for init_name in ['zeros', 'xavier']:
        for r in sorted(results[init_name]['step_records'], key=lambda x: x['step']):
            table.add_data(
                init_name,
                int(r['step']),
                float(r['analysis/grad_b_l0_n0']),
                float(r['analysis/grad_b_l0_n1']),
                float(r['analysis/grad_b_l0_n2']),
                float(r['analysis/grad_b_l0_n3']),
                float(r['analysis/grad_b_l0_n4']),
                float(r['analysis/grad_std_5_neurons']),
                float(r['train/loss_step']),
            )

    run.log(
        {
            'analysis/gradient_trace_plot': wandb.Image(str(grad_plot_path)),
            'analysis/val_accuracy_comparison_plot': wandb.Image(str(val_plot_path)),
            'analysis/gradient_trace_table': table,
        }
    )

    for init_name in ['zeros', 'xavier']:
        run.summary[f'{init_name}/run_id'] = results[init_name]['run_id']
        run.summary[f'{init_name}/final_val_accuracy'] = results[init_name]['final_val']['accuracy']
        run.summary[f'{init_name}/final_test_accuracy'] = results[init_name]['final_test']['accuracy']
        run.summary[f'{init_name}/final_test_f1'] = results[init_name]['final_test']['f1']
        run.summary[f'{init_name}/grad_std_mean_first50'] = float(
            np.mean([r['analysis/grad_std_5_neurons'] for r in results[init_name]['step_records']])
        )

    run.finish()
    return run.id


if RUN_EXPERIMENT:
    data = load_dataset(dataset=DATASET, seed=SEED)
    zeros = run_experiment('zeros', data)
    xavier = run_experiment('xavier', data)

    results = {'zeros': zeros, 'xavier': xavier}
    grad_path, val_path = make_plots(results)
    summary_run_id = log_summary(results, grad_path, val_path)

    payload = {
        'config': {
            'dataset': DATASET,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'loss': 'cross_entropy',
            'optimizer': 'sgd',
            'learning_rate': 0.01,
            'activation': 'sigmoid',
            'num_layers': 2,
            'hidden_size': [64, 64],
            'track_steps': TRACK_STEPS,
            'track_neurons': TRACK_NEURONS,
        },
        'runs': {
            'zeros': {
                'run_id': zeros['run_id'],
                'run_name': zeros['run_name'],
                'final_val': zeros['final_val'],
                'final_test': zeros['final_test'],
            },
            'xavier': {
                'run_id': xavier['run_id'],
                'run_name': xavier['run_name'],
                'final_val': xavier['final_val'],
                'final_test': xavier['final_test'],
            },
        },
        'summary_run': {'run_name': 'report_2_9_summary', 'run_id': summary_run_id},
        'artifacts': {
            'gradient_trace_plot': str(grad_path.relative_to(ROOT)),
            'val_accuracy_plot': str(val_path.relative_to(ROOT)),
        },
    }

    out_json = OUT_DIR / 'report_2_9_weight_init_symmetry.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Wrote', out_json)
else:
    print('RUN_EXPERIMENT=False -> no W&B calls made.')


In [ ]:
summary_path = ROOT / 'src' / 'tmp' / 'report_2_9_weight_init_symmetry.json'
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    print('Summary file:', summary_path)
    print('Summary run:', data['summary_run'])
    print('Zeros run:', data['runs']['zeros']['run_id'], 'test_acc=', round(data['runs']['zeros']['final_test']['accuracy'], 4))
    print('Xavier run:', data['runs']['xavier']['run_id'], 'test_acc=', round(data['runs']['xavier']['final_test']['accuracy'], 4))
else:
    print('No summary JSON yet. Set RUN_EXPERIMENT=True and run the previous cell.')
